<a href="https://colab.research.google.com/github/srk2695/banking-data-platform/blob/main/bronze_to_silver_pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **get the project and generate the raw data**

In [1]:
!git clone https://github.com/srk2695/banking-data-platform.git
%cd banking-data-platform
!pip install -q faker pyspark
!python src/generate_data.py

Cloning into 'banking-data-platform'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 48 (delta 16), reused 36 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 114.24 KiB | 19.04 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/banking-data-platform
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.4 MB/s eta 0:00:00
/content/banking-data-platform/src/generate_data.py:6: SyntaxWarning: invalid escape sequence '\g'
  python src\generate_data.py
Wrote to /content/banking-data-platform/data/bronze
  customers.csv        1,010 rows
  accounts.csv         1,960 rows
  transactions.csv    20,200 rows
Injected issues: nulls, duplicates, negative amounts, casing, orphan account IDs.


# **start Spark and read the bronze CSVs**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("banking").getOrCreate()

customers = spark.read.csv("data/bronze/customers.csv", header=True, inferSchema=True)
accounts = spark.read.csv("data/bronze/accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("data/bronze/transactions.csv", header=True, inferSchema=True)

print(customers.count(), accounts.count(), transactions.count())
transactions.printSchema()
transactions.show(5)

1010 1960 20200
root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_timestamp: timestamp (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)

+--------------+----------+-------------------+----------+------+--------+-----------------+-------+---------+
|transaction_id|account_id|      txn_timestamp|  txn_type|amount|currency|merchant_category|channel|   status|
+--------------+----------+-------------------+----------+------+--------+-----------------+-------+---------+
|    T000009030|  A0000167|2025-10-12 06:24:32|       FEE| 47.53|     USD|             NULL| MOBILE|COMPLETED|
|    T000012728|  A0001043|2025-10-12 06:47:45|WITHDRAWAL|196.47|     AED|    ENTERTAINMENT| ONLINE|COMPLETED|
|    T000018414|  A0001700|2025-10-12 07:32

# **customers and accounts**

In [4]:
from pyspark.sql import functions as F

silver_customers = (
    customers.dropDuplicates(["customer_id"])
    .select(
        "customer_id",
        F.initcap(F.trim("first_name")).alias("first_name"),
        F.initcap(F.trim("last_name")).alias("last_name"),
        F.lower(F.trim("email")).alias("email"),
        "phone",
        F.to_date("date_of_birth").alias("date_of_birth"),
        "city",
        "country",
        "segment",
        F.to_date("created_at").alias("created_at"),
    )
)

silver_accounts = accounts.select(
    "account_id",
    "customer_id",
    "account_type",
    "currency",
    F.to_date("opened_date").alias("opened_date"),
    F.upper(F.trim("status")).alias("status"),
    F.abs(F.col("balance").cast("double")).alias("balance"),
    (F.col("balance").cast("double") < 0).alias("balance_was_negative"),
)

print(silver_customers.count(), silver_accounts.count())

1000 1960


# **transactions and rejected rows**

In [5]:
acc_ids = silver_accounts.select("account_id").withColumn("acct_found", F.lit(True))

dedup = transactions.dropDuplicates(["transaction_id"])
joined = dedup.join(acc_ids, "account_id", "left")

silver_transactions = (
    joined.filter(F.col("acct_found").isNotNull() & F.col("amount").isNotNull())
    .select(
        "transaction_id",
        "account_id",
        F.col("txn_timestamp").cast("timestamp").alias("txn_timestamp"),
        F.upper(F.trim("txn_type")).alias("txn_type"),
        F.abs(F.col("amount").cast("double")).alias("amount"),
        (F.col("amount").cast("double") < 0).alias("amount_was_negative"),
        "currency",
        "merchant_category",
        "channel",
        F.upper(F.trim("status")).alias("status"),
    )
)

silver_rejected = (
    joined.filter(F.col("amount").isNull() | F.col("acct_found").isNull())
    .withColumn(
        "reject_reason",
        F.when(F.col("amount").isNull(), "NULL_AMOUNT").otherwise("ORPHAN_ACCOUNT"),
    )
    .drop("acct_found")
)

print(silver_transactions.count(), silver_rejected.count())

19740 260


# **validate**

In [6]:
print("bronze rows:     ", transactions.count())
print("bronze unique ids:", transactions.select("transaction_id").distinct().count())
print("silver good:     ", silver_transactions.count())
print("silver rejected: ", silver_rejected.count())

silver_rejected.groupBy("reject_reason").count().show()

dups = (silver_transactions.groupBy("transaction_id").count()
        .filter("count > 1").count())
print("duplicate ids in silver:", dups)

bronze rows:      20200
bronze unique ids: 20000
silver good:      19740
silver rejected:  260
+--------------+-----+
| reject_reason|count|
+--------------+-----+
|   NULL_AMOUNT|  200|
|ORPHAN_ACCOUNT|   60|
+--------------+-----+

duplicate ids in silver: 0


In [7]:
##save the output as Parquet
silver_customers.write.mode("overwrite").parquet("data/silver/customers")
silver_accounts.write.mode("overwrite").parquet("data/silver/accounts")
silver_transactions.write.mode("overwrite").parquet("data/silver/transactions")
silver_rejected.write.mode("overwrite").parquet("data/silver/transactions_rejected")